In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/training_sets/train_20251018_212601.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/3000_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_3000_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_3000_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_3000_train_500_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id    subreddit                                              title  \
 0  1kfpv1l        OBGYN       3 Questions about progesterone and menopause   
 1  1ljp5wz    BabyBumps                       Quit eating my f$#%ing food!   
 2  1kxz0r8  Miscarriage  1 MMC, 1 failed Misoprostol, Septic miscarriag...   
 3  1danfpl      therapy              Therapist red flag? Or doing his job?   
 4  1lknllw  Miscarriage       Anyone lose a healthy baby around 16 weeks?!   
 
                                             selftext          created_utc  \
 0  I’m in menopause. Had some post-meno bleeding ...  2025-05-05 23:11:44   
 1  On the verge of crashing out over this. My 30 ...  2025-06-24 22:34:25   
 2  At the end of April, my husband and I were at ...   2025-05-29 1:57:59   
 3  My (25F) partner (26M) sees a male CSAT (calli...  2024-06-07 22:17:33   
 4  \nI lost my baby girl at 16 weeks pregnant in ...   2025-06-26 1:34:22   
 
                                                  url 

In [2]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
r10_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r10_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
r10_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r10_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

In [3]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [4]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(r10_emb_A, r10_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_3000_train_500_test.json
